# Naif ajan (kapanma bilgisiz) - taban

Bilgi kazancini gosterir. **Mesafe farki ve U-donusu cezasi AYRI verilmeli** - kazancin %94,8'i varsayimdan geliyor (planda not edildi).

In [ ]:
# === COLAB KURULUM ===  (yerelde calistirinca atlanir)
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and not os.path.exists('/content/QuakeMind'):
    !git clone --depth 1 https://github.com/UtBird/QuakeMind.git /content/QuakeMind
if IN_COLAB:
    %cd /content/QuakeMind/model_benchmarks/06_rota
    %pip install -q osmnx pyrosm networkx
print('cwd:', os.getcwd())

## 1. Graf

In [ ]:
import sys; from pathlib import Path; sys.path.insert(0, str(Path.cwd()))
import common as C
C.CONFIG.update(dict(
    place='Antakya, Hatay, Turkey',   # veya 'Kahramanmaras, Turkey'
    n_od_pairs=120,                    # senaryo: Antakya 126, Maras 35
))
G = C.load_graph()
closures = C.load_closures()
od = C.sample_od_pairs(G, C.CONFIG['n_od_pairs'], C.CONFIG['seed'])
print('dugum/kenar:', G.number_of_nodes(), G.number_of_edges(), ' kapanma:', len(closures))

## 2. Kapanmasiz rota (mode='none')

In [ ]:
H0 = C.apply_damage(G, closures, C.CONFIG, mode='none')
import pickle; base = pickle.load(open('data/baseline_paths.pkl', 'rb')) if Path('data/baseline_paths.pkl').exists() else None
metrics, routes = C.evaluate_routes(H0, od, algo='astar', baseline_paths=base)
# naif ajan gercekte kapali kenarlara girer -> fiili basarisizligi ayrica olc:
Hreal = C.apply_damage(G, closures, C.CONFIG, mode='hard')
actual_fail = sum(1 for (s,t), r in zip(od, routes) if r['ok'] and not C.route_once(Hreal, s, t)['ok'])
metrics['actual_blocked_rate'] = actual_fail / len(od)
metrics

In [ ]:
C.save_result('naive_agent', metrics)